# **1) Setup**


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

In [ ]:
# use GPU if we have one (Runtime > Change runtime type > T4 GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# fix the seeds so results are reproducible
torch.manual_seed(42)
np.random.seed(42)

# **2) Download dataset & Preprocessing**
- Apply normalization using the mean and standard deviation of the MNIST dataset to make training faster and more stable.
- For the training set only, apply light data augmentation (slight rotations and shifts) to help the model adapt to variations in handwriting styles.
- Do not apply augmentation to the validation and test sets, ensuring a fair evaluation.

In [ ]:
mean, std = 0.1307, 0.3081   # MNIST stats

# transforms for training (with a bit of augmentation) and for eval (clean)
train_tf = transforms.Compose([
    transforms.RandomAffine(degrees=10, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize((mean,), (std,)),
])
eval_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((mean,), (std,)),
])

In [ ]:
# downloads automatically the first time
train_aug   = datasets.MNIST("./data", train=True,  download=True, transform=train_tf)
train_clean = datasets.MNIST("./data", train=True,  download=True, transform=eval_tf)
test_ds     = datasets.MNIST("./data", train=False, download=True, transform=eval_tf)

In [ ]:
print("Train images:", len(train_clean), "| Test images:", len(test_ds))

# **3) Exploratory Data Analysis (EDA)**

In [ ]:
# look at a few raw samples
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(train_clean.data[i], cmap="gray")
    ax.set_title(f"Label: {train_clean.targets[i].item()}")
    ax.axis("off")
plt.suptitle("Sample MNIST digits")
plt.show()

In [ ]:
# class distribution - checking if the data is balanced
counts = torch.bincount(train_clean.targets)
plt.figure(figsize=(8, 3))
plt.bar(range(10), counts.numpy(), color="steelblue")
plt.xticks(range(10))
plt.xlabel("Digit")
plt.ylabel("Count")
plt.title("Class distribution (train)")
plt.show()
print(counts.tolist())

## **4) Train / Validation split + DataLoaders**
We will split the `60000` into `55000` for training and `5000` for validation.

In [ ]:
# random split of indices (same indices for the aug and clean versions)
g = torch.Generator().manual_seed(42)
perm = torch.randperm(len(train_clean), generator=g).tolist()
train_idx, val_idx = perm[:55000], perm[55000:]

train_ds = Subset(train_aug, train_idx)     # augmented
val_ds   = Subset(train_clean, val_idx)     # clean

In [ ]:
batch_size = 128
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size)
test_loader  = DataLoader(test_ds,  batch_size=batch_size)

In [ ]:
print("Train:", len(train_ds), "| Val:", len(val_ds), "| Test:", len(test_ds))

## **5) Building Models**
**MLP:** Flattens the image (784 pixels) and passes it through fully connected layers. It is simple, but it loses the spatial structure of the image.

**CNN:** Uses convolutional layers to capture edges and shapes (lines, curves), making it much more accurate for images.

In [ ]:
# baseline: simple MLP
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 10),
        )
    def forward(self, x):
        return self.net(x)

In [ ]:
# the main model: CNN with BatchNorm and Dropout
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),                      # 28 -> 14
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),                      # 14 -> 7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(128, 10),
        )
    def forward(self, x):
        return self.classifier(self.features(x))


In [ ]:
models = {"MLP": MLP(), "CNN": CNN()}

for name, m in models.items():
    n_params = sum(p.numel() for p in m.parameters())
    print(f"{name}: {n_params:,} parameters")

## **6) Training**
- Loss: `CrossEntropyLoss`
- Optimizer: `Adam` (lr = 0.001) with `StepLR` to reduce the learning rate midway
- The best-performing model is saved based on validation accuracy
- The same loop trains both models, one after the other

In [ ]:
criterion = nn.CrossEntropyLoss()
epochs = 10
histories = {}

In [ ]:
for name, model in models.items():
    print(f"\n===== Training {name} =====")
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_acc = 0.0

    for epoch in range(epochs):
        # ---- train ----
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
        train_loss, train_acc = running_loss / total, correct / total

        # ---- validation ----
        model.eval()
        running_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                running_loss += loss.item() * images.size(0)
                correct += (outputs.argmax(1) == labels).sum().item()
                total += labels.size(0)
        val_loss, val_acc = running_loss / total, correct / total

        scheduler.step()

        history["train_loss"].append(train_loss); history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss);     history["val_acc"].append(val_acc)

        # keep the best checkpoint
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f"{name.lower()}_best.pth")

        print(f"Epoch {epoch+1:02d}/{epochs} | "
              f"train loss {train_loss:.4f} acc {train_acc:.4f} | "
              f"val loss {val_loss:.4f} acc {val_acc:.4f}")

    histories[name] = history
    print(f"Best val acc for {name}: {best_val_acc:.4f}")

## **7) Learning Curves**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for name, h in histories.items():
    axes[0].plot(h["val_loss"], label=f"{name} val")
    axes[0].plot(h["train_loss"], "--", label=f"{name} train")
    axes[1].plot(h["val_acc"], label=f"{name} val")
    axes[1].plot(h["train_acc"], "--", label=f"{name} train")
axes[0].set_title("Loss");     axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()
plt.show()

## **8) Evaluation on the Test Set**
We load the best checkpoint for each model and evaluate it on the 10,000 images that the model has never seen before.

In [ ]:
results = {}
all_preds = {}
all_labels = None

In [ ]:
for name, model in models.items():
    model.load_state_dict(torch.load(f"{name.lower()}_best.pth", map_location=device))
    model.eval()

    preds, labels_list = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model(images.to(device))
            preds.append(outputs.argmax(1).cpu())
            labels_list.append(labels)
    preds = torch.cat(preds).numpy()
    all_labels = torch.cat(labels_list).numpy()

    all_preds[name] = preds
    results[name] = (preds == all_labels).mean()

comparison = pd.DataFrame({
    "Model": list(results.keys()),
    "Test Accuracy": [f"{v*100:.2f}%" for v in results.values()],
    "Parameters": [sum(p.numel() for p in models[n].parameters()) for n in results],
})
comparison

In [ ]:
# confusion matrix for the CNN
fig, ax = plt.subplots(figsize=(7, 7))
cm = confusion_matrix(all_labels, all_preds["CNN"])
ConfusionMatrixDisplay(cm, display_labels=range(10)).plot(ax=ax, cmap="Blues", colorbar=False)
plt.title("CNN - Confusion Matrix")
plt.show()

print(classification_report(all_labels, all_preds["CNN"], digits=4))

## **9) Error Analysis**
Let’s examine the images that the CNN misclassified. You will often find errors such as 4↔9, 3↔5, or 7↔1—some of which even a human might find confusing.

In [ ]:
wrong_idx = np.where(all_preds["CNN"] != all_labels)[0]
print(f"CNN misclassified {len(wrong_idx)} out of {len(all_labels)} test images")

fig, axes = plt.subplots(2, 8, figsize=(14, 4.5))
for ax, idx in zip(axes.flat, wrong_idx[:16]):
    ax.imshow(test_ds.data[idx], cmap="gray")
    ax.set_title(f"True: {all_labels[idx]}\nPred: {all_preds['CNN'][idx]}", color="red", fontsize=9)
    ax.axis("off")
plt.suptitle("Misclassified examples (CNN)")
plt.show()

## **10) Testing with Real Handwritten Digits**
This is the most important step in the project. MNIST images are **centered** (the digit is in the middle, sized approximately `20×20` within a `28×28` frame), so we must apply the same preprocessing to any new image:
1. Convert to grayscale.
2. Ensure the digit is white against a black background (invert the colors if the background is light).
3. Crop the image to the boundaries of the digit.
4. Resize it to 20 pixels and center it within a 28×28 frame.
5. Normalize.

In [ ]:
# helper that turns any drawing/photo of a digit into an MNIST-style tensor
# (a function here because we reuse it in the Gradio demo below)
def prepare_digit(img_array):
    arr = np.array(img_array).astype(np.float32)

    # collapse to a single "ink" channel (digit bright, background dark)
    if arr.ndim == 3 and arr.shape[2] == 4:
        # RGBA canvas: flatten it onto a white background first
        alpha = arr[..., 3:4] / 255.0
        arr = arr[..., :3] * alpha + 255.0 * (1 - alpha)

    if arr.ndim == 3:
        ink = arr[..., :3].mean(axis=2)
    else:
        ink = arr

    if ink.mean() > 127:                         # light background -> invert
        ink = 255 - ink

    # crop to the bounding box of the digit
    ys, xs = np.where(ink > 40)
    if len(ys) == 0:
        return None
    ink = ink[ys.min():ys.max() + 1, xs.min():xs.max() + 1]

    # resize so the longest side is 20px, keep the aspect ratio
    h, w = ink.shape
    scale = 20.0 / max(h, w)
    new_h, new_w = max(1, int(round(h * scale))), max(1, int(round(w * scale)))
    small = Image.fromarray(ink.astype(np.uint8)).resize((new_w, new_h), Image.LANCZOS)

    # paste in the center of a 28x28 black canvas
    canvas = np.zeros((28, 28), dtype=np.float32)
    top, left = (28 - new_h) // 2, (28 - new_w) // 2
    canvas[top:top + new_h, left:left + new_w] = np.array(small)

    tensor = torch.from_numpy(canvas / 255.0)
    tensor = (tensor - mean) / std
    return tensor.unsqueeze(0).unsqueeze(0)      # shape (1, 1, 28, 28)

In [ ]:
# upload a photo / drawing of a digit (works in Colab)
try:
    from google.colab import files
    uploaded = files.upload()

    cnn = models["CNN"]
    cnn.eval()

    for fname in uploaded.keys():
        img = Image.open(fname).convert("RGB")
        x = prepare_digit(img)
        if x is None:
            print(f"{fname}: couldn't find a digit in the image")
            continue

        with torch.no_grad():
            probs = torch.softmax(cnn(x.to(device)), dim=1)[0].cpu()
        pred = probs.argmax().item()

        fig, axes = plt.subplots(1, 2, figsize=(8, 3))
        axes[0].imshow(img); axes[0].set_title("Original"); axes[0].axis("off")
        axes[1].imshow(x[0, 0] * std + mean, cmap="gray")
        axes[1].set_title(f"Model sees -> Pred: {pred} ({probs[pred]*100:.1f}%)")
        axes[1].axis("off")
        plt.show()
except ImportError:
    print("Not running in Colab - skip this cell or load an image manually with PIL.")

## **11) Interactive Demo with Gradio**
Draw a digit using your mouse (or your finger on a mobile device), and the model will predict it in real-time, along with the confidence level for each digit.

In [ ]:
import gradio as gr

cnn = models["CNN"].to(device)
cnn.eval()

def predict_drawing(canvas):
    if canvas is None:
        return None
    # newer gradio versions return a dict from the sketchpad
    if isinstance(canvas, dict):
        canvas = canvas.get("composite")
        if canvas is None:
            return None

    x = prepare_digit(canvas)
    if x is None:
        return None

    with torch.no_grad():
        probs = torch.softmax(cnn(x.to(device)), dim=1)[0].cpu().numpy()
    return {str(i): float(probs[i]) for i in range(10)}

demo = gr.Interface(
    fn=predict_drawing,
    inputs=gr.Sketchpad(label="Draw a digit (0-9)"),
    outputs=gr.Label(num_top_classes=3, label="Prediction"),
    title="Handwritten Digit Recognition",
    description="Draw a digit and the CNN will predict it.",
    live=True,
)
demo.launch(share=True)

## **11) Save Models**

In [ ]:
# final model for deployment / GitHub
torch.save(models["CNN"].state_dict(), "mnist_cnn.pth")
print("Saved mnist_cnn.pth")

# to load it later:
# model = CNN(); model.load_state_dict(torch.load("mnist_cnn.pth")); model.eval()